# Instancia Premier League 2026-2027

Descarga desde Football-Data la base de esta temporada y la usa como conjunto de equipos para resolver un calendario doble Round-Robin nuevo con el modelo compartido `modelo_calendario.py`. La fuente puede actualizarse durante la temporada.

## Qué representa esta instancia

El CSV histórico de 2026-2027 se usa para identificar los clubes disponibles de la temporada en curso. El modelo comprueba que detecte 20 equipos, pero no exige 380 partidos históricos: crea desde cero el calendario doble Round-Robin de 38 jornadas. Por eso puede ejecutarse con una base parcial si los 20 equipos ya están representados; el resultado no reproduce las jornadas oficiales.

## Flujo del notebook

1. Localiza el proyecto e importa `resolver_instancia` desde `modelo_calendario.py`.
2. Descarga la base de Football-Data y obtiene de ella los equipos.
3. Resuelve dos configuraciones: Gurobi por defecto y sin cortes ni heurísticas.
4. Resume estado, objetivo, cota, gap, tiempo y nodos, y muestra el calendario generado y una muestra de los datos históricos.

Las restricciones operativas opcionales no tienen efecto mientras sus listas estén vacías. El indicador de `viajes` solo representa partidos de visita en jornadas críticas, no distancias reales.

## Licencia para ejecutar

Antes de ejecutar la celda Python que resuelve el modelo, activa la licencia desde PowerShell o la terminal de Windows con el comando que entrega Gurobi: `grbgetkey CLAVE_ENTREGADA_POR_GUROBI` (reemplaza el texto por la clave real, sin escribirla en el notebook). Después reinicia el kernel y ejecuta las celdas. Si `grbgetkey` no se reconoce, instala Gurobi Optimizer para Windows o agrega su carpeta `bin` al `PATH`. La lectura del CSV por sí sola no requiere licencia.

In [ ]:
# Evaluar la instancia 2026-2027 con GurobiPy en dos configuraciones.
from pathlib import Path
import sys

import pandas as pd
import time
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2026-2027"
URL_CSV = str((Path.cwd().resolve().parent / "BASES" / "2026-2027.csv"))
PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}

TIEMPO_TOTAL = 3600
MARGEN_SEGUNDOS = 30
inicio = time.monotonic()
resultados = {}

for indice, (nombre, desactivar_cortes_heuristicas) in enumerate(configuraciones.items()):
    pendientes = len(configuraciones) - indice
    disponible = TIEMPO_TOTAL - (time.monotonic() - inicio) - MARGEN_SEGUNDOS

    if disponible < 1:
        print(f"Se omite {nombre}: se agotó el presupuesto de tiempo.")
        break

    tiempo_limite = max(1, int(disponible / pendientes))
    print(f"{nombre}: límite de Gurobi de {tiempo_limite} segundos.")

    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        tiempo_limite=tiempo_limite,
        **PARAMETROS_MODELO,
    )

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
reporte["Objetivo (ObjVal)"] = reporte["Objetivo (ObjVal)"].fillna("No factible")
reporte["Gap (MIPGap)"] = reporte["Gap (MIPGap)"].fillna("No factible")
estado_map = {
    1: "OPTIMAL",
    2: "INFEASIBLE",
    3: "INF_OR_UNBD",
    4: "CUTOFF",
    5: "LOADED",
    6: "UNKNOWN",
}
reporte["Estado Gurobi"] = reporte["Estado Gurobi"].map(lambda v: estado_map.get(v, v))
display(reporte)

resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos históricos descargados: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))

Set parameter Username
Set parameter LicenseID to value 2834871
Academic license - for non-commercial use only - expires 2027-06-12
Set parameter TimeLimit to value 3600
Set parameter OutputFlag to value 1
2026-2027: 20 equipos, 38 jornadas, 16,680 variables y 7,080 restricciones.
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  3600

Optimize a model with 7080 rows, 16680 columns and 71800 nonzeros (Min)
Model fingerprint: 0x1921d8ff
Model has 1480 linear objective coefficients
Variable types: 1480 continuous, 15200 integer (15200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Presolve removed 2960 rows and 0 columns
Presolve time: 0.40s

,Configuración,Estado Gurobi,Objetivo (ObjVal),Cota (ObjBoundC),Gap (MIPGap),Tiempo (s),Nodos (NodeCount)
0,Normal (Gurobi por defecto),9,42.0,5.142857,0.857143,3600.099,1829.0
1,Sin cortes ni heurísticas,9,NaN,5.142857,NaN,24911.797,996.0


Partidos históricos descargados: 50
Breaks: 42.0 | Viajes críticos: 0


,Jornada,Local,Visitante
0,1,Arsenal,Aston Villa
1,1,Bournemouth,Brentford
2,1,Brighton,Chelsea
3,1,Coventry,Crystal Palace
4,1,Everton,Fulham
...,...,...,...
375,38,Hull,Brighton
376,38,Ipswich,Bournemouth
377,38,Man United,Newcastle
378,38,Nott'm Forest,Liverpool


,Date,HomeTeam,AwayTeam
0,21/08/2026,Arsenal,Coventry
1,22/08/2026,Hull,Man United
2,22/08/2026,Everton,Crystal Palace
3,22/08/2026,Ipswich,Sunderland
4,22/08/2026,Nott'm Forest,Leeds
5,22/08/2026,Brentford,Tottenham
6,23/08/2026,Brighton,Aston Villa
7,23/08/2026,Man City,Bournemouth
8,23/08/2026,Newcastle,Liverpool
9,24/08/2026,Fulham,Chelsea
